# Matrix Transform

## Rotation

In [33]:
import torch
import math

theta = 45
rotation_2d = torch.tensor([
    [math.cos(theta), -math.sin(theta)], [math.sin(theta), math.cos(theta)]
])

rotation_3d_by_x = torch.tensor([
    [1, 0, 0],
    [0, math.cos(theta), -math.sin(theta)],
    [0, math.sin(theta), math.cos(theta)]
])

rotation_3d_by_y = torch.tensor([
    [math.cos(theta), 0, -math.sin(theta)],
    [0, 1, 0],
    [-math.sin(theta), 0, math.cos(theta)]
])

rotation_3d_by_z = torch.tensor([
    [math.cos(theta), -math.sin(theta), 0],
    [math.sin(theta), math.cos(theta), 0],
    [0, 0, 1]
])

## Scale

In [34]:
import torch

# This means scale X component by 0.6, and scale Y component by 2.0
scale_2d = torch.tensor([
    [0.6, 0.0], [0.0, 2.0]
])


## Shear

In [35]:
import torch

"""
|                       /
|       Shear into     /
|                     /
"""
shear_2d = torch.tensor([
    [1, 2], [0, 1]
])

print(f"Shear (1, 0) is: {shear_2d @ torch.tensor([1, 0])}")
print(f"Shear (0, 1) is: {shear_2d @ torch.tensor([0, 1])}")

Shear (1, 0) is: tensor([1, 0])
Shear (0, 1) is: tensor([2, 1])


## Reflection

In [36]:
import torch

"""
      |
*     |      *
      |
  Reflect Axis
"""
reflect_2d = torch.tensor([
    [-1, 0], [0, 1]
])

print(f"Reflect (1, 0) is: {reflect_2d @ torch.tensor([1, 0])}")

Reflect (1, 0) is: tensor([-1,  0])


## Order does matters

Result differs from:
1. first rotation, then scale
2. first scale, then rotation

In [39]:
base = torch.tensor([3.0, 4.0])

print(f"Order1 is: {
    rotation_2d.to(torch.float) @ 
    scale_2d.to(torch.float) @ 
    shear_2d.to(torch.float) @ 
    reflect_2d.to(torch.float) @
    base}")

print(f"Order2 is: {
    scale_2d.to(torch.float) @ 
    rotation_2d.to(torch.float) @ 
    shear_2d.to(torch.float) @ 
    reflect_2d.to(torch.float) @ 
    base}")

Order1 is: tensor([-5.2313,  6.7553])
Order2 is: tensor([-0.4662, 12.7116])


## Eigen Vector and Eigen Value

绝大多数向量被矩阵作用后会改变方向。特征向量很特别：矩阵只缩放它们，从不旋转它们。那个缩放因子就是特征值。

In [47]:
A = torch.tensor([
    [2, 1], [1, 2]
]).to(torch.float)

eigen_values, eigen_vectors = torch.linalg.eig(A)

eigen_values = eigen_values.to(torch.float)
eigen_vectors = eigen_vectors.to(torch.float)

print(f"Eigen values: {eigen_values}")
print(f"Eigen vectors: {eigen_vectors}")

print(f"eigen_vectors[:, 0] = {eigen_vectors[:, 0]}")
print(f"A @ eigen_vectors[:, 0] = {A @ eigen_vectors[:, 0]}")
print(f"Scaled factor = {eigen_values[0]}")

Eigen values: tensor([3., 1.])
Eigen vectors: tensor([[ 0.7071, -0.7071],
        [ 0.7071,  0.7071]])
eigen_vectors[:, 0] = tensor([0.7071, 0.7071])
A @ eigen_vectors[:, 0] = tensor([2.1213, 2.1213])
Scaled factor = 3.0


## Eigen Decomposition

In [ ]:
"""

A = V @ D @ V^(-1)

V = matrix whose columns are eigenvectors
D = diagonal matrix of eigenvalues
V^(-1) = inverse of V

"""

特征值为什么重要

PCA。 协方差矩阵的特征向量就是主成分。特征值告诉你每个成分捕获了多少方差。按特征值排序，保留前 k 个，你就得到了降维。

稳定性。 在循环网络和动力系统里，模大于 1 的特征值会让输出爆炸。模小于 1 则让它们消失。这就是用一句话说清楚的梯度消失/爆炸问题。

谱方法。 图神经网络用邻接矩阵的特征值。谱聚类用拉普拉斯矩阵的特征值。特征向量揭示了图的结构。

行列式作为体积缩放因子

变换矩阵的行列式告诉你它把面积（二维）或体积（三维）缩放了多少。

In [48]:
import pandas as pd

df = pd.DataFrame({
    "Determinant": [1, 2, 0, 1],
    "Result": ["Area perserved", "Area doubled", "Space crushed to lower dimension", "Area perserved but flipped"],
})
df

,Determinant,Result
0,1,Area perserved
1,2,Area doubled
2,0,Space crushed to lower dimension
3,1,Area perserved but flipped


# Code Practice

### Custom Transform Matrix

In [49]:
import math

def rotation_2d(theta):
    c, s = math.cos(theta), math.sin(theta)
    return [[c, -s], [s, c]]

def scaling_2d(x, y):
    return [[x, 0], [0, y]]

def shearing_2d(x, y):
    return [[1, x], [1, y]]

def reflection_x():
    return [[1, 0], [0, -1]]

def reflection_y():
    return [[-1, 0], [0, 1]]

def mat_vec_mul(matrix, vector):
    return [
        sum(matrix[i][j] * vector[j] for j in range(len(vector)))
        for i in range(len(matrix))
    ]

def mat_mul(a, b):
    rows_a, cols_b = len(a), len(b[0])
    cols_a = len(a[0])

    return [
        [
            sum(a[i][k] * b[k][j] for k in range(cols_a))
            for j in range(cols_b)
        ]
        for i in range(rows_a)
    ]

point = [1.0, 0.0]
angle = math.pi / 4

rotated = mat_vec_mul(rotation_2d(angle), point)
print(f"Rotate (1,0) by 45 deg: ({rotated[0]:.4f}, {rotated[1]:.4f})")

scaled = mat_vec_mul(scaling_2d(2, 3), [1.0, 1.0])
print(f"Scale (1,1) by (2,3): ({scaled[0]:.1f}, {scaled[1]:.1f})")

sheared = mat_vec_mul(shearing_2d(1, 0), [1.0, 1.0])
print(f"Shear (1,1) kx=1: ({sheared[0]:.1f}, {sheared[1]:.1f})")

reflected = mat_vec_mul(reflection_y(), [2.0, 1.0])
print(f"Reflect (2,1) across y: ({reflected[0]:.1f}, {reflected[1]:.1f})")

Rotate (1,0) by 45 deg: (0.7071, 0.7071)
Scale (1,1) by (2,3): (2.0, 3.0)
Shear (1,1) kx=1: (2.0, 1.0)
Reflect (2,1) across y: (-2.0, 1.0)


### Combination of Matrix

In [50]:
R = rotation_2d(math.pi / 2)
S = scaling_2d(2, 0.5)

# Caution with orders
rotation_then_scale = mat_mul(S, R)
scale_then_rotation = mat_mul(R, S)

point = [1.0, 0.0]
result1 = mat_vec_mul(rotation_then_scale, point)
result2 = mat_vec_mul(scale_then_rotation, point)

print(f"Rotation 90 then sacle: ({result1[0]:.2f}, {result1[1]:.2f})")
print(f"Scale then Rotation 90: ({result2[0]:.2f}, {result2[1]:.2f})")

print(f"Same? {result1 == result2}")

Rotation 90 then sacle: (0.00, 0.50)
Scale then Rotation 90: (0.00, 2.00)
Same? False


### Calculation of Determinant

In [ ]:
from sympy import discriminant


def eigen_values_2x2(matrix):
    a, b = matrix[0]
    c, d = matrix[1]

    trace = a + d
    det = a * d - b * c
    discriminant = trace ** 2 - 4 * det

    if discriminant < 0:
        real = trace / 2
        imag = (-discriminant) ** 0.5 / 2
        return (complex(real, imag), complex(real, -imag))
    sqrt_disc = discriminant ** 0.5
    return ((trace + sqrt_disc) / 2, (trace - sqrt_disc) / 2)

def eigen_vectors_2x2(matrix, eigen_value):
    a, b = matrix[0]
    c, d = matrix[1]

    if abs(b) > 1e-10:
        v = [b, eigen_value - a]
    elif abs(c) > 1e-10:
        v = [eigen_value - d, c]
    else:
        if abs(a - eigen_value) < 1e-10:
            v = [1, 0]
        else:
            v = [0, 1]
    mag = (v[0] ** 2 + v[1] ** 2) ** 0.5
    return [v[0]/mag, v[1]/mag]

A = [[2, 1], [1, 2]]

vals = eigen_values_2x2(A)
print(f"Matrix: {A}")
print(f"Eigenvalues: {vals[0]:.4f}, {vals[1]:.4f}")

for val in vals:
    vec = eigen_vectors_2x2(A, val)
    result = mat_vec_mul(A, vec)
    scaled = [val * vec[0], val * vec[1]]
    print(f"  lambda={val:.1f}, v={[round(x,4) for x in vec]}")
    print(f"    A@v = {[round(x,4) for x in result]}")
    print(f"    l*v = {[round(x,4) for x in scaled]}")